In [1]:
# ============================================================
# POUR Missing Experiment Result Checker (FULL VERSION)
# ============================================================

import os
import glob
import time
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, clear_output

pd.set_option("display.max_columns", 999)
pd.set_option("display.width", 2000)

ROOT = "./missing_experiments_bong3"

# ============================================================
# 1. TEST RESULT FILE SEARCH
# ============================================================

print("=" * 100)
print("SEARCHING TEST RESULT FILES")
print("=" * 100)

test_files = sorted(
    glob.glob(f"{ROOT}/**/test_result.csv", recursive=True)
)

for f in test_files:
    print(f)

print("\nTOTAL TEST RESULTS:", len(test_files))

# ============================================================
# 2. BUILD SUMMARY
# ============================================================

rows = []

for f in test_files:

    try:
        df = pd.read_csv(f)

        row = df.iloc[0].to_dict()

        exp = os.path.basename(os.path.dirname(f))

        row["exp"] = exp

        # ----------------------------------------------------
        # group tagging
        # ----------------------------------------------------

        if exp.startswith("mask_"):

            row["group"] = "mask"

            try:
                row["mask_ratio"] = float(exp.replace("mask_", ""))
            except:
                row["mask_ratio"] = None

            row["variant"] = "FULL"

        elif exp.startswith("ablation_"):

            row["group"] = "ablation"
            row["mask_ratio"] = None
            row["variant"] = exp.replace("ablation_", "")

        else:

            row["group"] = "other"
            row["mask_ratio"] = None
            row["variant"] = exp

        rows.append(row)

    except Exception as e:
        print("[SKIP]", f, e)

summary = pd.DataFrame(rows)

# ============================================================
# 3. SAVE SUMMARY
# ============================================================

summary_path = f"{ROOT}/ALL_RESULT_SUMMARY_FIXED.csv"

summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n[SAVED]", summary_path)

# ============================================================
# 4. SHOW SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("FULL RESULT SUMMARY")
print("=" * 100)

if len(summary) > 0:

    cols = [
        "exp",
        "group",
        "variant",
        "mask_ratio",
        "mae",
        "rmse",
        "uncertainty_error_corr",
    ]

    cols = [c for c in cols if c in summary.columns]

    display(
        summary[cols].sort_values("mae")
    )

else:
    print("NO RESULTS FOUND")

# ============================================================
# 5. MASK ROBUSTNESS
# ============================================================

print("\n" + "=" * 100)
print("MASK ROBUSTNESS")
print("=" * 100)

mask_df = summary[summary["group"] == "mask"].copy()

if len(mask_df) > 0:

    mask_df = mask_df.sort_values("mask_ratio")

    display(mask_df)

    # --------------------------------------------------------
    # plot
    # --------------------------------------------------------

    plt.figure(figsize=(7, 5))

    plt.plot(
        mask_df["mask_ratio"],
        mask_df["mae"],
        marker="o",
        linewidth=2,
        label="MAE"
    )

    plt.plot(
        mask_df["mask_ratio"],
        mask_df["rmse"],
        marker="s",
        linewidth=2,
        label="RMSE"
    )

    plt.xlabel("Mask Ratio")
    plt.ylabel("Error")
    plt.title("POUR Robustness under Partial Observation")

    plt.grid(True)
    plt.legend()

    plt.tight_layout()

    fig_path = f"{ROOT}/CHECK_mask_robustness_curve.png"

    plt.savefig(fig_path, dpi=200)

    plt.show()

    print("\n[SAVED]", fig_path)

else:

    print("NO MASK RESULTS FOUND")

# ============================================================
# 6. ABLATION
# ============================================================

print("\n" + "=" * 100)
print("ABLATION")
print("=" * 100)

abl_df = summary[summary["group"] == "ablation"].copy()

if len(abl_df) > 0:

    abl_df = abl_df.sort_values("mae")

    display(abl_df)

    # --------------------------------------------------------
    # MAE plot
    # --------------------------------------------------------

    plt.figure(figsize=(8, 5))

    plt.bar(
        abl_df["variant"],
        abl_df["mae"]
    )

    plt.ylabel("MAE")
    plt.title("Ablation Study")

    plt.xticks(rotation=20)

    plt.tight_layout()

    fig_path = f"{ROOT}/CHECK_ablation_mae.png"

    plt.savefig(fig_path, dpi=200)

    plt.show()

    print("\n[SAVED]", fig_path)

    # --------------------------------------------------------
    # uncertainty corr
    # --------------------------------------------------------

    if "uncertainty_error_corr" in abl_df.columns:

        plt.figure(figsize=(8, 5))

        plt.bar(
            abl_df["variant"],
            abl_df["uncertainty_error_corr"]
        )

        plt.ylabel("Corr(|error|, uncertainty)")
        plt.title("Uncertainty Calibration")

        plt.xticks(rotation=20)

        plt.tight_layout()

        fig2 = f"{ROOT}/CHECK_ablation_uncertainty.png"

        plt.savefig(fig2, dpi=200)

        plt.show()

        print("\n[SAVED]", fig2)

else:

    print("NO ABLATION RESULTS FOUND")

# ============================================================
# 7. TRAIN HISTORY CHECK
# ============================================================

print("\n" + "=" * 100)
print("TRAIN HISTORY")
print("=" * 100)

hist_files = sorted(
    glob.glob(f"{ROOT}/**/train_history.csv", recursive=True)
)

for f in hist_files:

    print("\n" + "-" * 80)
    print(f)
    print("-" * 80)

    try:

        h = pd.read_csv(f)

        print("\nLAST 5 EPOCHS")

        display(h.tail())

        if "val_mae" in h.columns:

            best_idx = h["val_mae"].idxmin()

            print("\nBEST EPOCH")

            display(h.loc[[best_idx]])

    except Exception as e:

        print(e)

# ============================================================
# 8. BEST RESULT
# ============================================================

print("\n" + "=" * 100)
print("BEST RESULT")
print("=" * 100)

if len(summary) > 0:

    best = summary.sort_values("mae").iloc[0]

    display(최고)

# ============================================================
# 9. GENERATED FIGURES
# ============================================================

print("\n" + "=" * 100)
print("GENERATED FIGURES")
print("=" * 100)

pngs = sorted(
    glob.glob(f"{ROOT}/**/*.png", recursive=True)
)

for p in pngs:
    print(p)

print("\nTOTAL PNG:", len(pngs))

# ============================================================
# 10. CURRENT RUNNING PROCESS
# ============================================================

print("\n" + "=" * 100)
print("CURRENT PYTHON PROCESS")
print("=" * 100)

os.system('ps -ef | grep "bong3.py" | grep -v grep')

# ============================================================
# 11. GPU STATUS
# ============================================================

print("\n" + "=" * 100)
print("GPU STATUS")
print("=" * 100)

os.system("nvidia-smi")

# ============================================================
# 12. LIVE MONITOR OPTION
# ============================================================

print("\n" + "=" * 100)
print("OPTIONAL LIVE MONITOR")
print("=" * 100)

print("""
실시간 monitoring 하고 싶으면 아래 셀 따로 실행:

while True:
    clear_output(wait=True)

    files = glob.glob("./missing_experiments_bong3/**/train_history.csv", recursive=True)

    for f in files:
        print("="*80)
        print(f)

        try:
            df = pd.read_csv(f)
            display(df.tail(3))
        except Exception as e:
            print(e)

    time.sleep(10)
""")

SEARCHING TEST RESULT FILES

TOTAL TEST RESULTS: 0


OSError: Cannot save file into a non-existent directory: 'missing_experiments_bong3'